In [0]:
# Load Config
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/notebooks/00_framework_config.ipynb

# Load Utilities
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/xsd_parser.ipynb
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/datatype_mapper.ipynb
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/schema_registry.ipynb
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/ddl_generator.ipynb
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/table_creator.ipynb

# Input
xsd_file = "v1_orders.xsd"
xsd_path = f"{XSD_VOLUME}/{xsd_file}"
table_name = xsd_file.replace(".xsd", "")

# Parse
metadata_df = extract_metadata(xsd_path)

# Map Types
mapped_df = map_xsd_to_sql(metadata_df)

# Registry
registry_df = enrich_registry_metadata(
    metadata_df=mapped_df,
    xsd_file=xsd_file,
    root_element="Orders",
    target_table=table_name,
    schema_version=1
)

write_to_registry(
    registry_df,
    SCHEMA_REGISTRY_TABLE
)

# Generate DDL

ddl = generate_ddl(
    metadata_df=registry_df,
    catalog=CATALOG,
    schema=BRONZE_SCHEMA,
    table_name=table_name
)
print(ddl)

# Create Table
create_table(ddl)
print(
    f"Table Created: "
    f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
)

In [0]:
spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.v1_orders"
).printSchema()